# Reddit

There is a [subreddit dedicated to the topic of Coronavirus](https://www.reddit.com/r/Coronavirus/). It currently has 1.5 million members.

> On December 2019, a novel coronavirus strain (SARS-CoV-2) emerged in the city of Wuhan, China. This subreddit seeks to monitor the spread of the disease COVID-19, declared a pandemic by the WHO. Please be civil and empathetic. This subreddit is for high quality posts and discussion.

Despite this description, this subreddit actually started in on May 3, 2013! Yep, coronaviruses have been with us for some time.

Is it possible to look through the URLs being shared there to see if there are seeds for the COVID-19 project? Let's install [praw](https://praw.readthedocs.io/en/latest/) the Python Reddit API Wrapper and find out.

In [ ]:
! pip --quiet install praw

To use praw you'll need API keys. Go over to https://www.reddit.com/prefs/apps and create an app. You can copy/paste them directly into your notebook below, but I've set mine in the environment for my Jupyter notebook to keep them private while sharing this notebook.

In [ ]:
import os
import praw

reddit = praw.Reddit(
    client_id = os.environ.get('REDDIT_CLIENT_ID')
    client_secret = os.environ.get('REDDIT_CLIENT_SECRET')
    user_agent = 'praw-edsu')

Let's look specifically at the Coronavirus subreddit:

In [ ]:
covid19 = reddit.subreddit('Coronavirus') 

## Posts

Reddit started as a site to share links to things, vote and comment on them. It attempts to rank these posts using what it calls "hottness" which is a ranking based on the votes it has received and its age. At one point in time the source code for Reddit was available, and it was possible to [definitively say](https://medium.com/hacking-and-gonzo/how-reddit-ranking-algorithms-work-ef111e33d0d9) how the algorithm worked. But now it's not entirely clear. At any rate the Reddit API provides access to the hottest posts.

In [ ]:
posts = covid19.hot(limit=10)

for post in posts:
    print(post.title)
    print(post.url)
    print("")

## Archived URLs

Now lets take a look at the top 100 stories, and see how many URLs have been archived at the Internet Archive.

In [ ]:
from wayback import WaybackClient
from wayback.exceptions import WaybackException

wb = WaybackClient()

checked = []
archived = []
errors = []

for post in covid19.hot(limit=25):
    checked.append(post.url)
    try:
        versions = wb.search(post.url)
        if len(list(versions)) > 0:
            archived.append(post.url)
    except WaybackException:
        errors.append(post.url)
    except Exception as e:
        print(e)
    
print('{0} URLs checked ; {1} archived ({2:.1f}%) ; {3} unable to be archived.'.format(
    len(checked),
    len(archived),
    len(archived) / len(checked) * 100,
    len(errors)
))

Ok, so what URLs have already been archived?

In [ ]:
for url in archived:
    print(url)

We can also see which ones have not been archived:

In [ ]:
for url in set(checked) - set(archived):
    print(url)

## PushShift API

Unfortunately, Reddit's API doesn't let you retrieve more than 100 of the hottest posts. But there is a service called [PushShift](https://pushift.io) that does make this data available via an [API] of their own. It appears that PushShift has [some kind of relationship](https://www.reddit.com/r/pushshift/comments/bcxguf/new_to_pushshift_read_this_faq/) with Reddit to make this possible, but it's not entirely clear what that relationship is.

So lets look the posts in the last hour on the Coronavirus subreddit using the PushShift API. The API is public (no authentication requied), and allows us to search the Coronavirus subreddit using time slices delimited by a `before` and `after` parameters.

In [ ]:
import requests

url = "https://api.pushshift.io/reddit/search/submission?subreddit=Coronavirus"
params = {
    "subreddit": "Coronavirus",
    "after": "1h",
    "limit": 1000
}

results = requests.get(url, params=params).json()['data']

In [ ]:
len(results)

PushShift make quite a bit more data available for each post--here's the first one:

In [ ]:
results[0]

For the purposes of appraising content about COVID-19 the `id`, `url`, `title`, `created_utc`, `score` and `num_comments` look like they could be useful. Let's create a function that will walk all the results back to January 1, 2020 (since we're interested in COVID-19) and write them to a CSV file. This is a bit of a longer stretch of code that I normally like to put in a Jupyter Notebook cell, but we need to do a bit of exception handling in case the API returns strangely.

A few things to note about this code. We start out fetching 1 hour windows with the `before` and `after` parameters. But as we encounter time slices that have no posts we expand this window using the `step` variable.

In [ ]:
import csv
import sys
import time
import datetime

out = csv.writer(open('data/reddit.csv', 'w'))
out.writerow(['id', 'url', 'title', 'created', 'creator', 'score', 'comments'])

url = "https://api.pushshift.io/reddit/search/submission?subreddit=Coronavirus"
params = {
    "subreddit": "Coronavirus",
    "limit": 1000
}

# keep track of the hour we are interested in
hour = 1

# step is the number of hours to look for at a time
step = 1

# calculate the number of hours since 2020-01-01
num_hours = (datetime.datetime.now() - datetime.datetime(2020, 1, 1)).total_seconds() / (60 * 60)

while hour < num_hours:
    sys.stdout.write('.')
    sys.stdout.flush()
    
    params['before'] = '{}h'.format(hour)
    params['after'] = '{}h'.format(hour + step)
    
    try:
        resp = requests.get(url, params=params)
        if resp.status_code != 200:
            print('received {} response'.format(resp.status_code))
        else:
            results = resp.json()['data']
            if len(results) > 0: 
                for result in results:
                    created = datetime.datetime.fromtimestamp(result['created_utc'])
                    out.writerow([
                        result['id'],
                        result['url'],
                        result['title'],
                        created.isoformat(),
                        result['author'],
                        result['score'],
                        result['num_comments']
                    ])

                # move the clock back further
                hour += step
            
            else:
                # we didn't find anything so increase the range
                step += 1
                        
    except Exception as e:
        print('uhoh: {}'.format(e))        

    # be polite and wait a little between requests
    time.sleep(0.5)

## Sorted

For manual inspection, and seeing what content changes over time it's actually useful to save the CSV sorted by the time it was created.

In [ ]:
import pandas
pandas.set_option('display.max_colwidth', None)

reddit = pandas.read_csv('data/reddit.csv', parse_dates=['created'], index_col='id')
reddit

Let's sort it by creation time:

In [ ]:
reddit = reddit.sort_values('created', ascending=True)
reddit

## Activity

Since there is so much activity it would be interesting to see what it looks like over time. But first we need to summarize it, because it's too many rows to pass off directly to Altair.

In [ ]:
import altair
altair.renderers.enable('html')

posts_by_hour = reddit.groupby('created').count()
posts_by_hour = posts_by_hour.resample('1D').sum()

posts_by_hour

altair.Chart(posts_by_hour.reset_index(), title="Coronavirus Subreddit Posts", width=800).mark_bar().encode(
    altair.X('monthdate(created)', title='Time (Days)'),
    altair.Y('id', title='Posts per Day')
)

## Domains

What are the top websites (domains) that are being submitted to?

In [ ]:
from urllib.parse import urlparse

reddit['domain'] = reddit['url'].map(lambda u: urlparse(u).netloc)
reddit.groupby('domain').count().sort_values('id', ascending=False)

There are lots of subdomains at the same domain. Let's see if the [tld](https://pypi.org/project/tld/) module can help:

In [ ]:
from tld import get_tld

def fld(url):
    tld = get_tld(url, as_object=True, fail_silently=True)
    return tld.fld if tld else ''

reddit['fld'] = reddit['url'].map(fld)
domains = reddit.groupby('fld').count().sort_values('url', ascending=False)
domains['url'].head(25)

In [ ]:
thehill = reddit[reddit['fld'] == 'thehill.com']
thehill

688 URLs from thehill.com?! Are they distinct?

In [ ]:
len(thehill['url'].unique())

Surprisingly, yes. How about the New York Times?

In [ ]:
nyt = reddit[reddit['fld'] == 'nytimes.com']
print(len(nyt['url']), 'total')
print(len(nyt['url'].unique()), 'unique')

How about ProPublica?

In [ ]:
propublica = reddit[reddit['fld'] == 'propublica.org']
propublica = propublica.sort_values('score', ascending=False)
print(len(propublica['url']), 'total')
print(len(propublica['url'].unique()), 'unique')
cols = ['created', 'creator', 'score', 'comments', 'url', 'title', 'fld']
df = propublica[cols]
df
#propublica = propublica.sort_values('votes', ascending=False)
#propublica = propublica['created', 'creator', 'score', 'comments', 'fld', 'url', 'title']
df.to_csv('propublica-reddit.csv')

## Users

Given some [recent discussion](https://twitter.com/MatthiasWhist/status/1243145523017797634) of how bots are influencing votes I thought it could be interesting to see what users are most present in this data, to try see if there is some obviously automated behavior.

In [ ]:
user_counts = reddit.groupby('creator').count()
user_counts = user_counts.sort_values('id', ascending=False)
user_counts['id'].head(50)

Does it seem kind of weird to you that the top posting user apparently has no posts on Reddit? 

https://www.reddit.com/user/KinnerNevada